# Test

### Bước 1: Tiền khởi tạo mô hình và tải dữ liệu Corpus gốc

In [ ]:
import os
import sys
from pprint import pprint

project_root = os.path.abspath(os.getcwd())
if project_root not in sys.path:
    sys.path.append(project_root)

from src.config import *
from src.utils import preprocess_text
from src.retrieval.demo_retrieval import load_news_corpus
from src.pipeline.evidence import prefetch_query_context, build_evidence_bundle, assess_with_llm
from src.llm.handler import get_llm
from src.slm.model import IntegratedSLM
from src.pipeline.selection import split_clean_noisy, finalize_remaining_noisy_with_slm

print('Đang khởi tạo các thành phần LLM và SLM...')
llm = get_llm()
slm = IntegratedSLM(model_path=MODEL_PATH, backend='hf')

print('Đang tải static corpus...')
static_corpus = load_news_corpus()
print(f'Số lượng tài liệu trong corpus: {len(static_corpus)}')

### Bước 2: Input mẫu tin và Tiền xử lý

In [ ]:
sample_claim = 'Bộ Giáo dục và Đào tạo cho học sinh nghỉ học thêm 1 học kỳ vì bão Yagi.'
ground_truth = 1 # Fake

print(f'Input thô: {sample_claim}')

# Tiền xử lý văn bản (bước đầu tiên mà mọi mẫu tin đều phải qua)
cleaned_text = preprocess_text(sample_claim)
print(f'Sau khi tiền xử lý: {cleaned_text}')

### Bước 3: Bootstrap - Truy xuất bối cảnh tri thức ban đầu 
Ở bước này, hệ thống gọi `prefetch_query_context()` để lấy thông tin giải nghĩa (Wikipedia/DuckDuckGo) và các tin tức mồi (Bing News) để phục vụ cho suốt vòng đời của `claim` này.

In [ ]:
print('Đang tải Query Context (Knowledge & Bing seed)...')

# Ép chế độ chỉ dùng Wikipedia và lấy dạng summary
knowledge_mode = 'wiki_only'
wiki_fetch_full = False

query_context = prefetch_query_context(
    text=cleaned_text,
    demo_k=TOP_K_DEMOS,
    fact_top_k=FACT_TOP_K,
    reuse_knowledge_cache=False,
    knowledge_mode=knowledge_mode,
    wiki_fetch_full=wiki_fetch_full,
)

print(f"\n-- Chế độ Kiến thức: {query_context['knowledge_mode']}")
print(f"\n-- Wikipedia fetch full: {wiki_fetch_full} (False = summary)")
print(f"\n-- Văn bản Tri thức (Knowledge text) thu được (500 kí tự đầu):\n", query_context['knowledge_text'][:500] + '...')
print(f"\n-- Số lượng tin bài mồi (Bing Seed News): {len(query_context['bing_seed_news'])}")
print("\n--- 1 Tin mồi ngẫu nhiên trong gói:\n")
pprint(query_context['bing_seed_news'][0] if len(query_context['bing_seed_news']) > 0 else 'Không có')

### Bước 4: Thiết lập Trạng thái (State) và Bắt đầu Round 1
Trong `runner.py`, mọi kết quả của một pipeline được quản lý qua một `state` dict.

In [ ]:
state = {
    "event_id": 0,
    "text": cleaned_text,
    "round": 0,
    "status": "unprocessed",
    "query_context": query_context,
    "ground_truth": ground_truth,
}

# Khởi tạo Pool lọc 
d_clean = []  # Những mẫu tin chắc chắn đã được dán nhãn đúng ở vòng này
d_noisy = []  # Những mẫu tin bị nhiễu do confidence bằng SLM thấp

round_id = 1
print(f"Bắt đầu xử lý cho Round {round_id}...")

#### 4.1 Xây dựng gói bằng chứng đặc trưng dựa theo vòng (`build_evidence_bundle`)

In [ ]:
# D_clean hiện tại bằng rỗng
demos, knowledge_k, retrieval_source = build_evidence_bundle(
    text=state["text"],
    static_corpus=static_corpus,
    clean_pool=d_clean,
    round_id=round_id,
    query_context=state["query_context"],
    demo_k=TOP_K_DEMOS,
)

print(f"Nguồn Retrieve: {retrieval_source}")
print(f"Số lượng Demos sử dụng: {len(demos)}")
print("Demo đầu tiên được đưa vào prompt cho LLM:\n")
pprint(demos[0] if demos else 'Không có demo')

#### 4.2 LLM tham gia đánh giá (`assess_with_llm`)

In [ ]:
print("Đang lấy đánh giá từ LLM (sẽ mất một chút thời gian)...")
assess = assess_with_llm(
    text=state["text"], 
    demos=demos, 
    knowledge_k=knowledge_k,
    llm=llm
)

# Cập nhật trạng thái state dựa theo đánh giá của LLM
state.update({
    "round": round_id,
    "label": assess["y_llm"],
    "label_llm": assess["y_llm"],
    "llm_raw": assess["llm_raw"],
    "retrieval_source": retrieval_source,
    "knowledge": knowledge_k,
    "prompt": assess["prompt"]  # prompt đã gửi đi cho llm
})

print(f"Nhãn do LLM phán đoán (0: Real, 1: Fake): {state['label_llm']}")
print("\n--- Đoạn text thô (raw) trả về từ LLM:\n", state['llm_raw'])

#### 4.3 SLM tiến hành ước lượng và đánh giá độ tin cậy để phân loại (`slm.inference`)

In [ ]:
print("Đưa văn bản đi qua bộ phân loại SLM...")
pred, conf, probs = slm.inference(state["text"])

state["label_slm"] = pred
state["y_slm"] = pred
state["conf_slm"] = conf

print(f"Nhãn dự đoán từ SLM: {pred}")
print(f"Độ tin cậy (Confidence): {conf:.4f}")
print(f"Mức xác suất: {probs}")

#### 4.4 Thực hiện chia nhãn thành Clean hay Noisy dựa vào độ tin cậy 
So sánh độ tin cậy của SLM với `CONFIDENCE_THRESHOLD`. Nếu >= thì cho vào bể sạch, ngược lại bị đem vào kho chờ xét ở Round tiếp.

In [ ]:
print(f"Ngưỡng tin cậy cấu hình (CONFIDENCE_THRESHOLD): {CONFIDENCE_THRESHOLD}")
is_clean = split_clean_noisy(state, CONFIDENCE_THRESHOLD)

if is_clean:
    state["status"] = "clean"
    d_clean.append(state)
    print("--- KẾT QUẢ: Mẫu tin này rõ ràng, đưa vào d_clean.")
else:
    state["status"] = "noisy"
    d_noisy.append(state)
    print("--- KẾT QUẢ: Mẫu tin nằm dưới mức tin cậy, không thể kết luận an toàn -> bị đưa vào d_noisy.")

### Bước 5: Mô phỏng Round 2 và Việc học Fine-tune (nếu có d_noisy)
Mục tiêu của MRCD là học hỏi sau mỗi vòng (Vòng lấy thông tin từ vòng trước để bù đáp cho kiến thức của mình).

In [ ]:
# MÔ PHỎNG: Để dễ test ở bước này, ta sẽ nạp vài mẫu tin "mock" giả lập clean pool 
dummy_clean_sample = {
    "text": preprocess_text("Đây là mẫu văn bản đã học và xác định đúng Real."),
    "label": 0,
    "round": 1
}
d_clean.append(dummy_clean_sample)

if len(d_noisy) > 0:
    round_id = 2
    print(f"--- Chuyển sang xử lý Round {round_id} cho tin Noisy")
    
    # SLM được học/finetune thêm từ d_clean đã tích lũy ở vòng trước 
    print(f"\n[a] Tiến hành Finetune lại RoBERTa SLM trên các {len(d_clean)} mẫu d_clean đã học:")
    ft_stats = slm.finetune_on_clean(
        clean_samples=d_clean,
        epochs=1,
        batch_size=8,
        lr=1e-5
    )
    pprint(ft_stats)
    
    # Ta build_evidence_bundle lại, lần này có D_clean tham gia (pool từ vòng trước)
    print("\n[b] Truy xuất dẫn chứng D_clean (Retrieval lần 2):")
    demos2, _, r_source2 = build_evidence_bundle(
        text=state["text"],
        static_corpus=static_corpus,
        clean_pool=d_clean, 
        round_id=round_id,
        query_context=state["query_context"],
        demo_k=TOP_K_DEMOS,
    )
    print(f"Nguồn lấy Demo vòng 2 là: {r_source2}")
    pprint(demos2)
    
    # ... tiếp đó lại chạy lặp lại bước 4.2 và 4.3 đối với các tin Noisy. Ở demo ta bỏ qua để tránh tốn token.
    
    # Nếu qua các round vẫn cứ là noisy thì bước cuối sẽ là final choice 
    print("\n[c] Final Judgment - Ép nhãn:")
    finalized_noisy = finalize_remaining_noisy_with_slm(d_noisy, slm)
    final_sample = finalized_noisy[0]
    final_sample["status"] = "finalized_by_slm"
    print(f"Chốt hạ trạng thái: {final_sample['status']}")
    print(f"Nhãn do mô hình ép buộc chốt lại: {final_sample['label']}")
else:
    print("Mẫu tin đã quá tự tin ở ngay vòng 1 (Clean), nên không cần xử lý Noisy flow !")